In [ ]:
import requests
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 1. Định nghĩa Schema
crypto_schema = T.StructType([
                                T.StructField("symbol", T.StringType(), True),
                                T.StructField("price", T.StringType(), True),
                                T.StructField("timestamp", T.StringType(), True)
                            ])

# 2. Hàm lấy dữ liệu từ Binance
symbols = ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'ADAUSDT', 'DOGEUSDT', 'PEPEUSDT', 'USDCUSDT']

def fetch_crypto_data():
    data_list = []
    for symbol in symbols:
        url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                data['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                data_list.append(data)
        except Exception as e:
            print(f"Lỗi khi lấy {symbol}: {e}")
    return data_list

# 3. Vòng lặp Ingestion
for i in range(25):
    print(f"Đang lấy dữ liệu đợt {i+1}...")
    
    # Lấy data thô
    raw_payload = fetch_crypto_data()
    
    # Tạo DataFrame với Schema đã định nghĩa 
    df_raw = spark.createDataFrame(raw_payload, schema=crypto_schema)
    
    # Chuyển datatype
    df_final = df_raw.withColumn("price", F.col("price").cast(T.DoubleType())) \
                     .withColumn("timestamp", F.to_timestamp(F.col("timestamp")))
    
    # Ghi vào Lakehouse
    df_final.write.mode("append").format("delta").saveAsTable("bronze_crypto_prices")
    
    time.sleep(30)


